# MIRAGE 재현(Colab) — 한 파일로: 환경 → 데이터 자동 설치 → 생성 스모크 테스트 → 학습 → 생성 → 시각화

저장소: https://github.com/MedARC-AI/MIRAGE (README 기준). 로컬에 GPU가 없어 **이 노트북은 Colab GPU에서 실제로 실행해 검증하지 못했습니다**. 각 단계 끝의 ✅/⚠️ 점검 셀이 막히는 곳을 알려 줍니다.

## 한 번에 실행하는 법
설정 셀의 `STAGES` 스위치로 단계를 고릅니다. 기본값은 **환경 + 데이터 + 생성 스모크 테스트까지**(학습 없이 GPU와 환경이 되는지 먼저 확인)이고, 학습과 생성은 `True`로 바꿔 다시 실행합니다.

## 자동으로 받는 것 (모두 명령으로 내려받음, 수동 업로드 없음)
- `reesekneeland/MIRAGE`: Stable Cascade 체크포인트(약 11GB), VDVAE, 캡션 npy
- `reesekneeland/umn-imagery`: NSD-Imagery 자극 `imagery_stimuli_18.pt`, `nsdimagery_stimuli.pkl3`, 전처리된 `preprocessed_data/subject{N}/nsd_imagery.pt`, 자극 CSV, nsdgeneral 마스크
- `pscotti/mindeyev2`(학습 단계에서만): 피험자 베타 약 1.9GB, COCO 224 이미지 22GB

## 먼저 알아둘 것
- **GPU:** README에 VRAM 요구량이 없습니다. 코드가 단계 C(약 7.2GB)와 단계 B(약 3.1GB)를 오프로딩 없이 올립니다. A100(40GB 이상) 권장, L4(24GB)는 빠듯할 수 있다는 것은 제 추정이며 스모크 테스트가 실제 사용량을 출력합니다.
- **RAM:** 학습 노트북이 학습 이미지를 float32로 올립니다. 고용량 RAM 런타임을 쓰세요.
- **Python:** MIRAGE는 torch 2.1.0 등을 고정합니다. `uv`로 Python 3.10 가상환경을 따로 만들어 그 안에서 실행합니다.
- **릿지 가중치:** MIRAGE용 사전학습 릿지 가중치(`ridge_*_weights.pkl`)는 공개 목록에서 찾지 못해 학습 단계에서 직접 학습합니다. (`reesekneeland/MIRAGE`의 `models/` 폴더의 `last.pth`는 다른 구조의 체크포인트로 보이며 이 파이프라인용인지 확인하지 못했습니다.)

## 확인하지 못한 것
1. `git_image_features.hdf5`(프롬프트 구성요소)를 끈 설정에서도 학습 노트북이 건너뛰는지.
2. 생성 결과 저장 형식(시각화 셀은 일반적인 (N,3,H,W) 텐서를 가정).
3. 정확한 학습 소요 시간.


In [ ]:
# 1. GPU / 메모리 / 디스크 점검
!nvidia-smi
!free -h | head -2
!df -h /content | tail -1

In [ ]:
# 2. 설정 (여기만 바꾸면 됩니다)
import os
ROOT        = "/content/MIRAGE"           # 저장소가 놓일 위치
SUBJ        = 1                           # 피험자 (1~8)
NUM_SESSIONS= 40                          # 학습 세션 수. 빠른 점검은 10, 논문 설정은 40
MODEL_NAME  = f"subj0{SUBJ}_{NUM_SESSIONS}sess_colab_ridge_sc2"
CAPTION_TYPE= "medium"                    # coco / short / medium / schmedium
USE_PROMPT_RECON = False                  # True면 GIT 캡션 프롬프트 사용(73k장 특징 계산: 매우 무겁다)
USE_BLURRY  = True                        # VDVAE 저수준 재구성
USE_DUAL_GUIDANCE = True
MODES       = ["imagery"]                 # "vision", "imagery", "shared1000"
GEN_REP     = 1                           # 기본 10. 첫 실행은 1
N_IMG       = 4                           # num_images_per_sample, 기본 16. 첫 실행은 4
NUM_TRIAL_REPS = 16                       # 기본 16
DRIVE_DIR   = "/content/drive/MyDrive/mirage_colab"  # 결과 백업(선택)
HF_TOKEN    = None                        # 필요하면 문자열로 지정

DATA   = f"{ROOT}/dataset"                # README의 ../dataset
CACHE  = f"{ROOT}/cache"                  # README의 ../cache
SRC    = f"{ROOT}/src"
VENV   = "/content/mei-env"
for d in (ROOT, DATA, f"{DATA}/preprocessed_data/subject{SUBJ}", CACHE):
    os.makedirs(d, exist_ok=True)
print(MODEL_NAME)
STAGES = {"setup": True, "download": True, "smoke": True, "train": False, "recon": False, "viz": False}
# 먼저 smoke까지만 True로 두고, 스모크 테스트가 통과하면 train, recon, viz를 차례로 True로 바꿔 다시 실행하세요.


In [ ]:
# 3. (선택) Google Drive 마운트: 체크포인트와 결과를 끊김에서 지키기 위한 백업용
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)

## A. 환경 설정 (Python 3.10 가상환경)
`setup.sh`의 고정 버전을 그대로 쓰되, `venv` 대신 `uv`로 Python 3.10을 준비합니다.

In [ ]:
%%bash
set -e
pip -q install uv
uv python install 3.10
if [ -x /content/mei-env/bin/python ]; then
  echo "가상환경이 이미 있어 재사용합니다 (새로 만들려면 rm -rf /content/mei-env 후 다시 실행)"
else
  uv venv /content/mei-env --python 3.10
fi
/content/mei-env/bin/python --version
# 이후 모든 설치가 지켜야 할 제약: 선택 패키지(wandb, evaluate 등)가 huggingface-hub 2.x 등으로 올려서 transformers 4.37.2가 깨지는 것을 막는다.
cat > /content/constraints.txt <<'CEOF'
huggingface-hub>=0.23.2,<1.0
numpy==1.26.3
torch==2.1.0
torchvision==0.16.0
transformers==4.37.2
diffusers==0.30.3
protobuf<5
setuptools<81
CEOF
echo "venv ok"


In [ ]:
%%bash
set -eo pipefail
PY=/content/mei-env/bin/python
# 핵심 패키지: setup.sh에 고정된 버전만 고정하고, 나머지(huggingface-hub, safetensors, timm, scipy, pillow)는 resolver가 정하게 둔다.
# (diffusers==0.30.3은 huggingface-hub>=0.23.2를 요구하므로 requirements.txt의 0.20.3을 함께 고정하면 충돌한다.)
uv pip install --python $PY -c /content/constraints.txt \
  numpy==1.26.3 matplotlib==3.8.2 tqdm scikit-image==0.22.0 accelerate==0.24.1 webdataset==0.2.73 \
  pandas==2.2.0 "einops>=0.7.0" ftfy regex kornia==0.7.1 h5py==3.10.0 open_clip_torch \
  torch==2.1.0 torchvision==0.16.0 transformers==4.37.2 xformers==0.0.22.post7 torchmetrics==1.3.0.post0 \
  diffusers==0.30.3 omegaconf==2.3.0 pytorch-lightning==2.0.1 sentence-transformers==2.5.1 \
  nibabel==5.2.1 munch==4.0.0 "setuptools<81" safetensors timm scipy pillow ipykernel jupyter nbconvert nbformat 2>&1 | tail -n 40
echo "core install done"


In [ ]:
%%bash
PY=/content/mei-env/bin/python
# setup.sh의 나머지(선택적/무거운 것들은 실패해도 계속). 필요한 import가 없다고 나오면 그때 다시 시도하세요.
uv pip install --python $PY -c /content/constraints.txt git+https://github.com/ildoonet/pytorch-gradual-warmup-lr.git || echo "WARN gradual-warmup"
uv pip install --python $PY -c /content/constraints.txt git+https://github.com/pabloppp/pytorch-tools || echo "WARN pytorch-tools"
uv pip install --python $PY -c /content/constraints.txt git+https://github.com/openai/CLIP.git --no-deps || echo "WARN CLIP"
uv pip install --python $PY -c /content/constraints.txt dalle2-pytorch || echo "WARN dalle2-pytorch"
uv pip install --python $PY -c /content/constraints.txt wandb==0.16.2 deepspeed==0.13.1 evaluate==0.4.1 nltk==3.8.1 rouge_score==0.1.2 || echo "WARN optional"
uv pip install --python $PY -c /content/constraints.txt insightface==0.7.3 opencv-python==4.8.1.78 onnxruntime==1.16.3 onnx2torch==1.5.13 || echo "WARN insightface/onnx (추론에는 불필요할 가능성이 큼, 확인 안 함)"
uv pip install --python $PY -c /content/constraints.txt "huggingface-hub>=0.23.2,<1.0" "protobuf<5" || echo "WARN 제약 재적용 실패"
$PY -m ipykernel install --user --name mei-env --display-name "mei-env (py3.10)"
$PY -c "import torch, transformers, diffusers; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('transformers', transformers.__version__, 'diffusers', diffusers.__version__)"


In [ ]:
# 4. 저장소와 StableCascade 클론 (README: src/StableCascade/ 위치)
!git clone -q https://github.com/MedARC-AI/MIRAGE.git {ROOT}_repo || echo "이미 있음"
!rsync -a {ROOT}_repo/ {ROOT}/
!git clone -q https://github.com/Stability-AI/StableCascade.git {SRC}/StableCascade || echo "이미 있음"
!ls {SRC} | head -30

## B. 체크포인트와 데이터 받기 (필요한 파일만)
- 체크포인트(`reesekneeland/MIRAGE`): 단계 A/B/C, effnet, previewer, VDVAE.
- 캡션(`reesekneeland/MIRAGE`): 학습용 캡션 .npy.
- NSD 베타·COCO 이미지(`pscotti/mindeyev2`): 피험자 베타 약 1.9GB, COCO 224 이미지 22GB.


In [ ]:
# 5. 데이터 자동 설치 (Hugging Face)
%pip -q install huggingface_hub
from huggingface_hub import hf_hub_download, snapshot_download
if STAGES["download"]:
    # (1) Stable Cascade 체크포인트와 VDVAE: 생성에 필요
    for fn in ["effnet_encoder.safetensors", "previewer.safetensors", "stage_a.safetensors",
               "stage_b_bf16.safetensors", "stage_c_bf16.safetensors", "imagenet64-iter-1600000-model-ema.th"]:
        hf_hub_download("reesekneeland/MIRAGE", fn, repo_type="dataset", local_dir=CACHE, token=HF_TOKEN)
    # (2) NSD-Imagery 자극·전처리 베타·자극 CSV·nsdgeneral 마스크: 생성(imagery 모드)에 필요
    snapshot_download("reesekneeland/umn-imagery", repo_type="dataset", local_dir=DATA, token=HF_TOKEN, allow_patterns=[
        "nsddata_stimuli/stimuli/imagery_stimuli_18.pt", "nsddata_stimuli/stimuli/nsdimagery_stimuli.pkl3",
        "nsddata_stimuli/stimuli/imagery_images/*", f"preprocessed_data/subject{SUBJ}/nsd_imagery.pt",
        "preprocessed_data/captions_18.npy", "nsddata/experiments/nsd/nsd_stim_info_merged.csv",
        f"nsddata/ppdata/subj0{SUBJ}/func1pt8mm/roi/nsdgeneral.nii.gz"])
if STAGES["train"]:
    # (3) 학습에만 필요: 캡션, 피험자 베타, COCO 224 이미지(22GB, 시간이 오래 걸림)
    caps = {"coco": "annots_73k.npy", "short": "short_length_captions.npy", "medium": "mid_length_captions_73K.npy"}
    for k in {"coco", CAPTION_TYPE}:
        if k in caps:
            hf_hub_download("reesekneeland/MIRAGE", caps[k], repo_type="dataset", local_dir=f"{DATA}/preprocessed_data", token=HF_TOKEN)
    hf_hub_download("pscotti/mindeyev2", f"betas_all_subj0{SUBJ}_fp32_renorm.hdf5", repo_type="dataset", local_dir=DATA, token=HF_TOKEN)
    hf_hub_download("pscotti/mindeyev2", "coco_images_224_float16.hdf5", repo_type="dataset", local_dir=DATA, token=HF_TOKEN)
    hf_hub_download("pscotti/mindeyev2", "shared1000.npy", repo_type="dataset", local_dir=DATA, token=HF_TOKEN)
print("download stage done")

In [ ]:
# 6. 사전 점검: 단계별로 필요한 파일이 모두 있는지
ck = ["effnet_encoder.safetensors", "previewer.safetensors", "stage_a.safetensors", "stage_b_bf16.safetensors", "stage_c_bf16.safetensors", "imagenet64-iter-1600000-model-ema.th"]
need = [f"{CACHE}/{f}" for f in ck]
need += [f"{DATA}/nsddata_stimuli/stimuli/imagery_stimuli_18.pt", f"{DATA}/nsddata_stimuli/stimuli/nsdimagery_stimuli.pkl3",
         f"{DATA}/preprocessed_data/subject{SUBJ}/nsd_imagery.pt", f"{DATA}/nsddata/experiments/nsd/nsd_stim_info_merged.csv",
         f"{DATA}/nsddata_stimuli/stimuli/imagery_images/6.png"]
if STAGES["train"]:
    need += [f"{DATA}/betas_all_subj0{SUBJ}_fp32_renorm.hdf5", f"{DATA}/coco_images_224_float16.hdf5", f"{DATA}/preprocessed_data/annots_73k.npy"]
missing = [p for p in need if not os.path.exists(p)]
print("✅ 모두 있음" if not missing else "⚠️ 없음:\n" + "\n".join(missing))

## 의존성 자동 복구
MIRAGE와 StableCascade가 import하는 모듈을 실제로 불러 보고, 없는 모듈이 나오면 알려진 설치 방법으로 자동 설치한 뒤 다시 시도합니다(최대 15회). 같은 오류로 계속 멈추면 마지막 출력을 알려 주세요.

In [ ]:
# 7a. 의존성 점검과 자동 복구
import subprocess, re
PY = "/content/mei-env/bin/python"
SPEC = {
    "warmup_scheduler": ["git+https://github.com/ildoonet/pytorch-gradual-warmup-lr.git"],
    "torchtools": ["git+https://github.com/pabloppp/pytorch-tools"],
    "clip": ["--no-deps", "git+https://github.com/openai/CLIP.git"],
    "wandb": ["wandb==0.16.2"],
    "dalle2_pytorch": ["dalle2-pytorch"],
    "cv2": ["opencv-python==4.8.1.78"],
    "PIL": ["pillow"], "sklearn": ["scikit-learn"], "skimage": ["scikit-image==0.22.0"], "yaml": ["pyyaml"],
}
def uv_install(args):
    r = subprocess.run(["uv", "pip", "install", "--python", PY, "-c", "/content/constraints.txt"] + args, capture_output=True, text=True)
    print(("OK   " if r.returncode == 0 else "FAIL ") + " ".join(args)); 
    if r.returncode: print(r.stderr[-600:])
    return r.returncode == 0
probe = f'''
import sys, os
sys.path.insert(0, "{SRC}"); sys.path.insert(0, "{SRC}/StableCascade"); os.chdir("{SRC}")
import torch
import utils, vdvae, modeling_git, sc_reconstructor
print("imports ok")
'''
open("/content/_probe.py", "w").write(probe)
for attempt in range(15):
    r = subprocess.run([PY, "/content/_probe.py"], capture_output=True, text=True, cwd=SRC)
    if r.returncode == 0:
        print("✅", r.stdout.strip()); break
    m = re.search(r"No module named '([^'.]+)", r.stderr)
    if not m:
        print("⚠️ 모듈 누락이 아닌 오류:\n" + r.stderr[-1500:]); break
    mod = m.group(1)
    print("누락:", mod)
    if not uv_install(SPEC.get(mod, [mod.replace("_", "-")])):
        break
else:
    print("⚠️ 15회 안에 해결 못 함")

## 생성 스모크 테스트 (학습 없이)
정답 자극(자연 장면)의 **진짜 CLIP 임베딩**으로 Stable Cascade가 이미지를 만들 수 있는지 확인합니다. 환경, 체크포인트, GPU 메모리가 되는지만 보는 점검이며 뇌 데이터는 쓰지 않습니다. 사용한 GPU 메모리 최댓값을 출력합니다.

In [ ]:
# 7. 생성 스모크 테스트
if STAGES["smoke"]:
    test_py = f'''
import sys, os, time, torch
sys.path.insert(0, "{SRC}"); sys.path.insert(0, "{SRC}/StableCascade")
os.chdir("{SRC}")
from PIL import Image
from sc_reconstructor import SC_Reconstructor
t0 = time.time()
rec = SC_Reconstructor(device="cuda:0", cache_dir="{CACHE}", embedder_only=False, compile_models=False)
print("모델 로드", round(time.time()-t0), "초, GPU 메모리(GB):", round(torch.cuda.max_memory_allocated()/1e9, 1))
outs = []
for k in (6, 7):   # 세트 B 자연 장면 자극 중 2장
    im = Image.open("{DATA}/nsddata_stimuli/stimuli/imagery_images/%d.png" % k).convert("RGB")
    c_i = rec.embed_image(im)
    try:
        c_t = rec.embed_text(["a photo"])
    except Exception as e:
        print("embed_text 호출 실패(시그니처 확인 필요):", repr(e)); raise
    t1 = time.time()
    out = rec.reconstruct(c_i=c_i, c_t=c_t, n_samples=1, strength=1.0, num_steps_c=10, num_steps_b=5)
    print("자극", k, "생성", round(time.time()-t1), "초, 출력", tuple(out.shape), "GPU 최대(GB):", round(torch.cuda.max_memory_allocated()/1e9, 1))
    outs.append((im, out[0].float().cpu()))
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axs = plt.subplots(2, len(outs), figsize=(4*len(outs), 8))
for i, (im, o) in enumerate(outs):
    axs[0, i].imshow(im); axs[0, i].axis("off"); axs[0, i].set_title("정답")
    axs[1, i].imshow(o.permute(1, 2, 0).numpy()); axs[1, i].axis("off"); axs[1, i].set_title("정답 임베딩에서 생성")
plt.tight_layout(); plt.savefig("{ROOT}/smoke_test.png", dpi=120)
print("저장:", "{ROOT}/smoke_test.png")
'''
    open("/content/_smoke.py", "w").write(test_py)
    !cd {SRC} && /content/mei-env/bin/python /content/_smoke.py
    from IPython.display import Image as _I, display
    if os.path.exists(f"{ROOT}/smoke_test.png"):
        display(_I(f"{ROOT}/smoke_test.png"))
else:
    print("STAGES['smoke']=False: 건너뜀")

## C. 학습 (릿지 회귀, perception 데이터만)
저장소의 `src/Train.ipynb`를 **인자만 바꿔서** 3.10 가상환경 커널로 실행합니다. 결과는 `train_logs/{MODEL_NAME}/ridge_*_weights.pkl`입니다.
- `USE_PROMPT_RECON=False`이면 GIT 캡션 프롬프트 구성요소를 끕니다(73k장 특징 계산 회피). 이 경우 노트북이 `git_image_features.hdf5` 존재 여부만 확인하는지 확인하지 못해 **작은 더미 파일**을 만들어 둡니다.
- 시간이 오래 걸립니다(README에 예상 시간 없음). 세션이 끊기면 `preprocessed_data/` 캐시를 재사용하므로 같은 셀을 다시 실행하면 이어집니다.

In [ ]:
# 10. 노트북 인자 패치 + 실행 도우미
import json, re, subprocess
def patch_notebook(src_nb, dst_nb, repl):
    nb = json.load(open(src_nb))
    hit = 0
    for c in nb["cells"]:
        if c["cell_type"] != "code":
            continue
        s = "".join(c["source"])
        if "jupyter_args" in s and "f\"" in s:
            s2 = re.sub(r'jupyter_args\s*=\s*f"[^"]*"', lambda m: "jupyter_args = " + repr(repl) , s, flags=re.S)
            if s2 != s:
                c["source"] = s2.splitlines(keepends=True)
                hit += 1
    json.dump(nb, open(dst_nb, "w"))
    print("patched cells:", hit)
    return hit
def run_nb(path, out):
    cmd = ["/content/mei-env/bin/jupyter", "nbconvert", "--to", "notebook", "--execute", path, "--output", out,
           "--ExecutePreprocessor.kernel_name=mei-env", "--ExecutePreprocessor.timeout=-1"]
    print(" ".join(cmd))
    r = subprocess.run(cmd, cwd=SRC)
    return r.returncode

In [ ]:
if STAGES['train']:
    # 11. 학습 실행
    import h5py
    git_path = f"{DATA}/git_image_features.hdf5"
    if not USE_PROMPT_RECON and not os.path.exists(git_path):
        with h5py.File(git_path, "w") as f:
            f.create_dataset("features", data=np.zeros((1,), np.float32))   # 더미: 프롬프트 구성요소를 끈 경우 건너뛰기 위함
    flags = ["--dual_guidance" if USE_DUAL_GUIDANCE else "--no-dual_guidance",
             "--prompt_recon" if USE_PROMPT_RECON else "--no-prompt_recon",
             "--blurry_recon" if USE_BLURRY else "--no-blurry_recon"]
    args = (f"--data_path=../dataset/ --cache_dir=../cache/ --model_name={MODEL_NAME} --batch_size=64 --no-multi_subject "
            f"--subj={SUBJ} --num_sessions={NUM_SESSIONS} --caption_type {CAPTION_TYPE} " + " ".join(flags))
    patch_notebook(f"{SRC}/Train.ipynb", f"{SRC}/Train_colab.ipynb", args)
    rc = run_nb("Train_colab.ipynb", "Train_colab_out.ipynb")
    print("return code", rc)
else:
    print("STAGES['train']=False: 건너뜀")

In [ ]:
# 12. 학습 결과 점검
tl = f"{ROOT}/train_logs/{MODEL_NAME}"
print(os.listdir(tl) if os.path.exists(tl) else "⚠️ train_logs가 없음: 위 실행 로그를 확인하세요")

## D. 생성 (재구성)
`src/recon_inference_mi.ipynb`를 인자만 바꿔서 실행합니다. 사전에 NSD-Imagery 베타를 `preprocessed_data/subject{N}/nsd_imagery.pt`로 만들어야 합니다(저장소의 `utils.create_whole_region_imagery_*`).

In [ ]:
if STAGES['recon']:
    # 15. 생성 실행 (모드마다)
    for MODE in MODES:
        flags = ["--dual_guidance" if USE_DUAL_GUIDANCE else "--no-dual_guidance",
                 "--prompt_recon" if USE_PROMPT_RECON else "--no-prompt_recon",
                 "--blurry_recon" if USE_BLURRY else "--no-blurry_recon"]
        args = (f"--data_path=../dataset --cache_dir=../cache --model_name={MODEL_NAME} --subj={SUBJ} --mode {MODE} "
                f"--gen_rep={GEN_REP} --num_images_per_sample={N_IMG} --num_trial_reps={NUM_TRIAL_REPS} " + " ".join(flags))
        patch_notebook(f"{SRC}/recon_inference_mi.ipynb", f"{SRC}/recon_{MODE}_colab.ipynb", args)
        print(MODE, "return code", run_nb(f"recon_{MODE}_colab.ipynb", f"recon_{MODE}_out.ipynb"))
    print(os.listdir(f"{ROOT}/evals/{MODEL_NAME}") if os.path.exists(f"{ROOT}/evals/{MODEL_NAME}") else "⚠️ evals가 없음")
else:
    print("STAGES['recon']=False: 건너뜀")

## E. 시각화와 재현 정도
정답(목표 자극), 저수준(blurry) 재구성, 최종 재구성을 나란히 보고, CLIP ViT-L/14 임베딩으로 **18개 자극 중 정답 찾기** 정확도를 계산합니다(우연 1/18). 저장 형식은 확인하지 못해 형태를 출력하고 일반적인 (N,3,H,W) 텐서를 가정합니다.

In [ ]:
if STAGES['viz']:
    # 16. 결과 불러오기와 그림
    import torch, matplotlib.pyplot as plt, numpy as np
    MODE = MODES[0]
    EV = f"{ROOT}/evals/{MODEL_NAME}"
    recons = torch.load(f"{EV}/{MODEL_NAME}_all_recons_{MODE}.pt", map_location="cpu")
    blurry = torch.load(f"{EV}/{MODEL_NAME}_all_blurryrecons_{MODE}.pt", map_location="cpu") if os.path.exists(f"{EV}/{MODEL_NAME}_all_blurryrecons_{MODE}.pt") else None
    gt = torch.load(f"{DATA}/nsddata_stimuli/stimuli/imagery_stimuli_18.pt", map_location="cpu")
    print("recons", getattr(recons, "shape", type(recons)), "blurry", getattr(blurry, "shape", None), "gt", gt.shape)
    def to_img(t):
        t = t.float()
        if t.min() < 0: t = (t + 1) / 2
        return t.clamp(0, 1).permute(1, 2, 0).numpy()
    n = min(len(recons), len(gt))
    rows = 3 if blurry is not None else 2
    fig, axs = plt.subplots(rows, n, figsize=(1.6 * n, 1.8 * rows))
    for i in range(n):
        axs[0, i].imshow(to_img(gt[i])); axs[0, i].axis("off")
        axs[1, i].imshow(to_img(recons[i])); axs[1, i].axis("off")
        if blurry is not None:
            axs[2, i].imshow(to_img(blurry[i])); axs[2, i].axis("off")
    axs[0, 0].set_title("정답", fontsize=8, loc="left"); axs[1, 0].set_title("재구성", fontsize=8, loc="left")
    plt.tight_layout(); plt.savefig(f"{ROOT}/recon_grid_{MODE}.png", dpi=150); plt.show()
else:
    print("STAGES['viz']=False: 건너뜀")

In [ ]:
%pip -q install transformers
if STAGES['viz']:
    # 17. CLIP 식별 정확도(18개 중 정답 찾기, 우연 1/18)
    from transformers import CLIPModel, CLIPProcessor
    dev = "cuda"
    cm = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(dev).eval()
    pr = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
    @torch.no_grad()
    def emb(ts):
        ims = [(to_img(t) * 255).astype(np.uint8) for t in ts]
        f = cm.get_image_features(**pr(images=ims, return_tensors="pt").to(dev))
        f = f.pooler_output if hasattr(f, "pooler_output") else f
        return torch.nn.functional.normalize(f, dim=-1).cpu()
    E_gt, E_rc = emb(gt[:n]), emb(recons[:n])
    S = E_rc @ E_gt.T
    top1 = (S.argmax(1) == torch.arange(n)).float().mean().item()
    top1b = (S.argmax(0) == torch.arange(n)).float().mean().item()
    print(f"재구성 → 정답 식별 top-1 {top1:.3f}, 정답 → 재구성 {top1b:.3f}, 우연 {1/n:.3f}")
    plt.figure(figsize=(4, 4)); plt.imshow(S.numpy(), cmap="viridis"); plt.colorbar(); plt.xlabel("정답"); plt.ylabel("재구성"); plt.savefig(f"{ROOT}/clip_sim_{MODE}.png", dpi=150); plt.show()
else:
    print("STAGES['viz']=False: 건너뜀")

In [ ]:
if STAGES['viz']:
    # 18. 결과 Drive 백업(선택)
    import shutil
    for p in (f"{ROOT}/train_logs/{MODEL_NAME}", f"{ROOT}/evals/{MODEL_NAME}"):
        if os.path.exists(p):
            shutil.copytree(p, f"{DRIVE_DIR}/{os.path.basename(os.path.dirname(p))}/{MODEL_NAME}", dirs_exist_ok=True)
    for f in (f"{ROOT}/recon_grid_{MODE}.png", f"{ROOT}/clip_sim_{MODE}.png"):
        if os.path.exists(f): shutil.copy(f, DRIVE_DIR)
    print("saved to", DRIVE_DIR)
else:
    print("STAGES['viz']=False: 건너뜀")